# Train UAV Detector (Thermal or RGB)

Notebook-first workflow for local machines, Colab, or SCC Jupyter. The same notebook can prepare Anti-UAV data for either `ir` or `rgb`, run a smoke test, run a parity training job, and compare metrics against the checked-in thermal weights.


In [ ]:
from pathlib import Path
import os
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

try:
    from src.utils.project import find_project_root
except ModuleNotFoundError:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            sys.path.insert(0, str(candidate))
            break
    from src.utils.project import find_project_root

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)
print('IN_COLAB =', IN_COLAB)


In [ ]:
import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA visible devices:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('Device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU 0:', torch.cuda.get_device_name(0))
    _probe = torch.zeros(1, device='cuda')
    print('CUDA probe tensor:', _probe)


In [ ]:
# Toggle these values before running.
MODALITY = 'rgb'  # 'ir' or 'rgb'
RUN_MODE = 'smoke'  # 'smoke' or 'parity'
INSTALL_REQUIREMENTS = False
DOWNLOAD_ANTI_UAV_RGBT = IN_COLAB and MODALITY == 'rgb'
ANTI_UAV_RGBT_URL = 'https://drive.google.com/file/d/1NPYaop35ocVTYWHOYQQHn8YHsM9jmLGr/view'
RUN_THERMAL_BASELINE_VALIDATION = True

RAW_DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
EXTRACT_ROOT = RAW_DATA_ROOT / 'Anti-UAV-RGBT'
RAW_TRAIN_DIR = RAW_DATA_ROOT / 'train'
RAW_TEST_DIR = None
OUTPUT_ROOT = PROJECT_ROOT / 'data' / ('rgb_uav' if MODALITY == 'rgb' else 'thermal_uav')
DATA_CONFIG = PROJECT_ROOT / 'configs' / ('dataset_rgb_uav.yaml' if MODALITY == 'rgb' else 'dataset_thermal_uav.yaml')
TRAIN_CONFIG = PROJECT_ROOT / 'configs' / ('train_detector_rgb.yaml' if MODALITY == 'rgb' else 'train_detector.yaml')
SPLIT_MANIFEST = RAW_DATA_ROOT / 'train_split_seed42.json'

print('MODALITY =', MODALITY)
print('RUN_MODE =', RUN_MODE)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('DATA_CONFIG =', DATA_CONFIG)
print('TRAIN_CONFIG =', TRAIN_CONFIG)


In [ ]:
import shutil
import subprocess

if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])

if DOWNLOAD_ANTI_UAV_RGBT:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])
    archive_path = RAW_DATA_ROOT / 'Anti-UAV-RGBT.zip'
    if not archive_path.exists():
        subprocess.check_call([sys.executable, '-m', 'gdown', '--fuzzy', ANTI_UAV_RGBT_URL, '-O', str(archive_path)])
    if not EXTRACT_ROOT.exists():
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(archive_path), str(EXTRACT_ROOT))

print('Raw data root:', RAW_DATA_ROOT)
print('Extract root:', EXTRACT_ROOT)


In [ ]:
def find_sequence_root_candidates(root: Path) -> list[Path]:
    if not root.exists():
        return []

    candidates: list[Path] = []
    probe_paths = [root, *root.rglob('*')]
    for path in probe_paths:
        if not path.is_dir():
            continue
        child_dirs = [child for child in path.iterdir() if child.is_dir()]
        if not child_dirs:
            continue
        if any(
            (child / 'IR_label.json').exists()
            or (child / 'RGB_label.json').exists()
            or (child / 'visible_label.json').exists()
            or (child / 'gt' / 'gt.txt').exists()
            for child in child_dirs
        ):
            candidates.append(path)
    return sorted(set(candidates))

if not RAW_TRAIN_DIR.exists():
    train_candidates = find_sequence_root_candidates(EXTRACT_ROOT)
    if len(train_candidates) == 1:
        RAW_TRAIN_DIR = train_candidates[0]
    elif train_candidates:
        print('Discovered candidate train roots:')
        for candidate in train_candidates:
            print(' -', candidate)

print('RAW_TRAIN_DIR =', RAW_TRAIN_DIR)
print('RAW_TEST_DIR =', RAW_TEST_DIR)
print('SPLIT_MANIFEST =', SPLIT_MANIFEST)


In [ ]:
prepare_cmd = [
    sys.executable, '-m', 'src.utils.prepare_anti_uav',
    '--raw-train-dir', str(RAW_TRAIN_DIR),
    '--output-root', str(OUTPUT_ROOT),
    '--modality', MODALITY,
    '--split-manifest', str(SPLIT_MANIFEST),
    '--val-ratio', '0.2',
]
if RAW_TEST_DIR:
    prepare_cmd.extend(['--raw-test-dir', str(RAW_TEST_DIR)])

print('Running:', ' '.join(prepare_cmd))
subprocess.check_call(prepare_cmd, cwd=PROJECT_ROOT)

check_cmd = [
    sys.executable, '-m', 'src.utils.dataset_checks',
    '--data', str(DATA_CONFIG),
]
print('Running:', ' '.join(check_cmd))
subprocess.check_call(check_cmd, cwd=PROJECT_ROOT)


In [ ]:
from src.detection.train import load_yaml, run_training

run_label = 'rgb_uav' if MODALITY == 'rgb' else 'thermal_uav'
run_name = f"yolo12n_{run_label}_smoke" if RUN_MODE == 'smoke' else f"yolo12n_{run_label}"

TRAIN_CFG = load_yaml(TRAIN_CONFIG)
TRAIN_CFG.update({
    'device': 'auto',
    'name': run_name,
})

if RUN_MODE == 'smoke':
    TRAIN_CFG.update({
        'epochs': 10,
        'imgsz': 640,
        'batch': -1,
        'workers': 2,
        'cache': 'disk',
        'amp': True,
        'verbose': True,
    })
else:
    TRAIN_CFG.update({
        'amp': True,
        'verbose': True,
    })

TRAIN_CFG


In [ ]:
save_dir = run_training(TRAIN_CFG, project_root=PROJECT_ROOT)
save_dir


In [ ]:
import json
import pandas as pd
from src.detection.validate import run_validation

val_cfg = {
    'weights': str(save_dir / 'weights' / 'best.pt'),
    'data': str(DATA_CONFIG),
    'split': 'val',
    'imgsz': TRAIN_CFG['imgsz'],
    'batch': TRAIN_CFG['batch'],
    'device': 'auto',
    'workers': TRAIN_CFG['workers'],
    'project': 'runs/val',
    'name': f"{TRAIN_CFG['name']}_val",
    'exist_ok': True,
}
metrics_file = run_validation(val_cfg, project_root=PROJECT_ROOT)
print('RGB/IR run metrics:', metrics_file)

comparison_rows = []
current_metrics = json.loads(Path(metrics_file).read_text(encoding='utf-8'))
comparison_rows.append({'model': MODALITY, **current_metrics})

thermal_metrics_file = None
if RUN_THERMAL_BASELINE_VALIDATION:
    thermal_val_cfg = {
        'weights': str(PROJECT_ROOT / 'weights' / 'best.pt'),
        'data': str(PROJECT_ROOT / 'configs' / 'dataset_thermal_uav.yaml'),
        'split': 'val',
        'imgsz': 960,
        'batch': 16,
        'device': 'auto',
        'workers': TRAIN_CFG['workers'],
        'project': 'runs/val',
        'name': 'thermal_checkedin_baseline_val',
        'exist_ok': True,
    }
    thermal_metrics_file = run_validation(thermal_val_cfg, project_root=PROJECT_ROOT)
    thermal_metrics = json.loads(Path(thermal_metrics_file).read_text(encoding='utf-8'))
    comparison_rows.append({'model': 'thermal_baseline', **thermal_metrics})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


Switch `RUN_MODE` from `smoke` to `parity` after the smoke test succeeds. The parity run keeps the same base model and nominal hyperparameters as the thermal baseline while still using `device='auto'` for Colab convenience.
